In [6]:
import pandas as pd
import numpy as np
import re
import pickle

import nltk
from nltk.corpus import stopwords



from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

In [7]:
df = pd.read_csv('data.csv')
df.head()

,clean_text,Sentiment
0,nice product good quality price rising bad sig...,1
1,didnt supplied yonex mavis outside cover yonex...,0
2,worst product damaged shuttlecock packed new b...,0
3,quite k nowadays quality cork like year back u...,1
4,pricedjust retaileri didnt understand wat adva...,0


In [8]:
print("Dataset Shape:", df.shape)
print("Columns:\n", df.columns)
print("Sample Data:\n", df.head())

Dataset Shape: (8510, 2)
Columns:
 Index(['clean_text', 'Sentiment'], dtype='object')
Sample Data:
                                           clean_text  Sentiment
0  nice product good quality price rising bad sig...          1
1  didnt supplied yonex mavis outside cover yonex...          0
2  worst product damaged shuttlecock packed new b...          0
3  quite k nowadays quality cork like year back u...          1
4  pricedjust retaileri didnt understand wat adva...          0


In [9]:
print("Rating Distribution:\n", df['Sentiment'].value_counts())

Rating Distribution:
 Sentiment
1    7438
0    1072
Name: count, dtype: int64


In [11]:
# CREATE SENTIMENT LABELS


def create_sentiment(rating):

    if rating >= 4:
        return 1
    elif rating <= 2:   
        return 0
    else:
        return np.nan

df['sentiment'] = df['Sentiment'].apply(create_sentiment)
df = df.dropna(subset=['sentiment'])


print("After Removing Neutral Reviews:", df.shape)

After Removing Neutral Reviews: (8510, 3)


In [13]:
# Data Cleaning
stop_words = stopwords.words('english')
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z ]', '', text) # Remove special characters
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

df['clean_review'] = df['clean_text'].apply(clean_text)


print("Cleaned Text Sample:\n", df['clean_review'].head())

Cleaned Text Sample:
 0    nice product good quality price rising bad sig...
1    didnt supplied yonex mavis outside cover yonex...
2    worst product damaged shuttlecock packed new b...
3    quite k nowadays quality cork like year back u...
4    pricedjust retaileri didnt understand wat adva...
Name: clean_review, dtype: object


In [20]:
# Check data
print(df.head())
print(df['Sentiment'].value_counts())

# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['clean_text'])   # ✅ Correct column
y = df['Sentiment']                         # ✅ Correct column

print("TF-IDF Shape:", X.shape)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train labels:\n", y_train.value_counts())

# Model Training
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

print("Model Training Completed")

# Prediction
y_pred = model.predict(X_test)

# Evaluation
f1 = f1_score(y_test, y_pred)
print("F1 Score:", f1)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

                                          clean_text  Sentiment  sentiment  \
0  nice product good quality price rising bad sig...          1          0   
1  didnt supplied yonex mavis outside cover yonex...          0          0   
2  worst product damaged shuttlecock packed new b...          0          0   
3  quite k nowadays quality cork like year back u...          1          0   
4  pricedjust retaileri didnt understand wat adva...          0          0   

                                        clean_review  
0  nice product good quality price rising bad sig...  
1  didnt supplied yonex mavis outside cover yonex...  
2  worst product damaged shuttlecock packed new b...  
3  quite k nowadays quality cork like year back u...  
4  pricedjust retaileri didnt understand wat adva...  
Sentiment
1    7438
0    1072
Name: count, dtype: int64
TF-IDF Shape: (8510, 3603)
Train labels:
 Sentiment
1    5950
0     858
Name: count, dtype: int64
Model Training Completed
F1 Score: 0.9553686934

In [21]:
pickle.dump(model, open('sentiment_model.pkl', 'wb'))
pickle.dump(tfidf, open('tfidf_vectorizer.pkl', 'wb'))


print("Model and Vectorizer Saved")

Model and Vectorizer Saved
